# 🌊 Automated Sentinel-1 & Sentinel-2 Flood Mapping Pipeline

This notebook automates the processing of Sentinel-1 & Sentinel-2 data to detect flood-affected areas.

## 📌 Steps Covered:

- **Batch Unzip & Extract Sentinel-1 GRD data**
- **Clip to AOI (Using a GeoJSON File)** 
- **Preprocess Each Image** (Radiometric Calibration, Terrain Correction)  
- **Convert to dB (VV/VH Polarization)**   
- **Apply Water Detection Algorithm** (Thresholding VV/VH)  
- **Compare Pre-Flood vs. Post-Flood to Detect Flooded Areas**  
- **Batch Unzip & Extract Sentinel-2 data**
- **Apply Water Detection Algorithm** (NDWI)  
- **Compare Pre-Flood vs. Post-Flood to Detect Flooded Areas**  
- **Fusion of Flooded Layers**
- **Statistical Analysis and Visulization**  


## Install Libraries

In [1]:
!pip install rasterio geopandas numpy matplotlib glob2 tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 18.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.6/323.6 kB 35.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 81.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 117.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 102.1 MB/s eta 0:00:00
  Created wheel for glob2: filename=glob2-0.7-py2.py3-none-any.whl size=9301 sha256=25e5823415b9e010760d233f962095c6d8d31c2a1c10ec4664458f6222df287d
  Stored in directory: /home/musmani/.cache/pip/wheels/37/07/ce/cbe8d31ad93224571b49fa03f8a5da11cdb31d3845ff73e0f3
Successfully built glob2


In [ ]:
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape
import digitalhub as dh
import zipfile
import os
from pathlib import Path
import matplotlib.pyplot as plt

## Access the platform and artifacts

In [3]:
import digitalhub as dh

# Define project name
project_name = "docker-sentinel"

# Access the project
project = dh.get_or_create_project(project_name)

# List all artifacts in the project
artifacts = project.list_artifacts()

# Print available artifacts
artifact_names = [artifact.name for artifact in artifacts]
print("Available Artifacts in Project:", artifact_names)

Available Artifacts in Project: ['sentinel2_post_flood', 'sentinel2_pre_flood', 'sentinel1_GRD_preflood', 'sentinel1_GRD_postflood']


In [5]:
print(dir(artifact))

['__add__', '__class__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getnewargs__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mod__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__rmod__', '__rmul__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'capitalize', 'casefold', 'center', 'count', 'encode', 'endswith', 'expandtabs', 'find', 'format', 'format_map', 'index', 'isalnum', 'isalpha', 'isascii', 'isdecimal', 'isdigit', 'isidentifier', 'islower', 'isnumeric', 'isprintable', 'isspace', 'istitle', 'isupper', 'join', 'ljust', 'lower', 'lstrip', 'maketrans', 'partition', 'removeprefix', 'removesuffix', 'replace', 'rfind', 'rindex', 'rjust', 'rpartition', 'rsplit', 'rstrip', 'split', 'splitlines', 'startswith', 'strip', 'swapcase', 'title', 'translate', 'upper', 'zfill']


In [4]:
project = dh.get_or_create_project('docker-sentinel')

In [7]:
af = project.get_artifact('sentinel1_GRD_preflood')
#af.download()

In [1]:

# Define project name
project_name = "docker-sentinel"
project = dh.get_or_create_project(project_name)

proj
# Define artifact names
artifacts = ["sentinel1_GRD_preflood", "sentinel1_GRD_postflood"]

# Define local download directory
download_dir = Path("./sentinel1_data")
download_dir.mkdir(exist_ok=True, parents=True)
# Access the project
project = dh.get_or_create_project(project_name)
# Function to download and extract artifacts
def download_and_extract(artifact_name):
    artifact = project.get_artifact(artifact_name)
    artifact_files = artifact.get_artifact()
    
    for file in artifact_files:
        if file.endswith(".SAFE.zip"):
            local_zip_path = download_dir / file
            
            # Download file if not already present
            if not local_zip_path.exists():
                print(f"Downloading {file}...")
                artifact.download(file, local_zip_path)
            
            # Extract file
            extract_path = download_dir / file.replace(".zip", "")
            if not extract_path.exists():
                print(f"Extracting {file}...")
                with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
                    zip_ref.extractall(download_dir)
            
            print(f"Processed {file} ✅")

# Process both pre-flood and post-flood artifacts
for artifact in artifacts:
    download_and_extract(artifact)

print("✅ All Sentinel-1 GRD data downloaded and extracted!")

AttributeError: 'ArtifactArtifact' object has no attribute 'get_artifact'

In [ ]:
import zipfile
import os
import glob

# Set directories
zip_folder = "Sentinel1_ZIP_Files"
unzipped_folder = "Processed_SAFE"

# Unzip all Sentinel-1 GRD files
zip_files = glob.glob(os.path.join(zip_folder, "*.zip"))

for zip_file in zip_files:
    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall(unzipped_folder)

print("✅ All Sentinel-1 files extracted.")


## Sentinel-2 data processing

In [ ]:
# Define pre-flood and post-flood folders
pre_flood_folder = "/content/Sentinel-2(Pre-NDWI)"
post_flood_folder = "/content/Sentinel-2(post-NDWI)"

# Find all NDWI TIFF files in each folder
pre_flood_files = sorted(glob.glob(os.path.join(pre_flood_folder, "*.tif")))
post_flood_files = sorted(glob.glob(os.path.join(post_flood_folder, "*.tif")))

print(f"Found {len(pre_flood_files)} pre-flood images and {len(post_flood_files)} post-flood images.")

# Function to visualize NDWI images
def visualize_ndwi(files, title_prefix):
    num_images = len(files)
    cols = 3
    rows = (num_images // cols) + (num_images % cols > 0)  # Auto calculate rows

    fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axs = axs.flatten()

    for i, file in enumerate(files):
        with rasterio.open(file) as src:
            ndwi = src.read(1)
        
        axs[i].imshow(ndwi, cmap='RdYlGn', vmin=-1, vmax=1)
        axs[i].set_title(f"{title_prefix} {i+1}")  # Simplified label
        axs[i].axis("off")
    
    # Hide any extra subplots
    for j in range(i+1, len(axs)):
        axs[j].axis("off")

    plt.tight_layout()
    plt.show()

# Visualize NDWI images before and after flood
visualize_ndwi(pre_flood_files, "NDWI Before")
visualize_ndwi(post_flood_files, "NDWI After")

In [ ]:
# Function to compute the mean NDWI across multiple images
def compute_mean_ndwi(files):
    ndwi_stack = []

    for file in files:
        with rasterio.open(file) as src:
            ndwi = src.read(1)
            ndwi_stack.append(ndwi)

    # Compute mean NDWI
    ndwi_stack = np.array(ndwi_stack)
    mean_ndwi = np.mean(ndwi_stack, axis=0)

    return mean_ndwi

# Compute mean NDWI for pre-flood and post-flood images
ndwi_pre = compute_mean_ndwi(pre_flood_files)
ndwi_post = compute_mean_ndwi(post_flood_files)

# Plot before and after NDWI side by side
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Before Flood
axs[0].imshow(ndwi_pre, cmap='RdYlGn', vmin=-1, vmax=1)
axs[0].set_title("Mean NDWI Before Flood")
axs[0].axis("off")

# After Flood
axs[1].imshow(ndwi_post, cmap='RdYlGn', vmin=-1, vmax=1)
axs[1].set_title("Mean NDWI After Flood")
axs[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Function to calculate water pixels and plot histograms
def analyze_water(ndwi_array, title, threshold=0):
    # Classify water pixels based on NDWI threshold
    water_pixels = ndwi_array > threshold
    water_percentage = np.sum(water_pixels) / water_pixels.size * 100  # Calculate percentage
    
    # Plot histogram of NDWI values
    plt.figure(figsize=(8, 6))
    plt.hist(ndwi_array.flatten(), bins=50, range=(-1, 1), color='blue', alpha=0.7)
    plt.axvline(x=threshold, color='red', linestyle='--', label='Water Threshold')
    plt.xlabel("NDWI Value")
    plt.ylabel("Pixel Count")
    plt.title(f"Histogram of NDWI - {title}")
    plt.legend()
    plt.show()

    print(f"Water area in {title}: {water_percentage:.2f}%")

# Analyze water for both images
analyze_water(ndwi_pre, "Before Flood", threshold=0)
analyze_water(ndwi_post, "After Flood", threshold=0)

In [ ]:
# Calculate NDWI difference
ndwi_diff = ndwi_post - ndwi_pre

# Apply flood threshold
flood_layer = (ndwi_diff > 0.1).astype(np.uint8)  # 1 = flooded, 0 = not flooded

# Optional: visualize the flood layer
plt.figure(figsize=(6, 6))
plt.imshow(flood_layer, cmap='Blues')
plt.title("Detected Flood Layer (S2)")
plt.axis("off")
plt.show()


## Downloading shapefile (flood layer-S2)

In [ ]:
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd

# Function to convert flood pixels (from NDWI difference) to shapefile
def convert_flood_to_shapefile(flood_array, transform, output_shapefile, raster_crs):
    # Convert binary flood mask to shapes (polygons)
    mask = flood_array.astype(np.uint8)  # Ensure it's uint8 for the shapes function
    results = shapes(mask, mask=mask, transform=transform)

    # Extract geometries
    geometries = [shape(geom) for geom, _ in results]

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame({'geometry': geometries}, crs=raster_crs)

    # Save to shapefile
    gdf.to_file(output_shapefile)
    print(f"✅ Flood shapefile saved to: {output_shapefile}")

# Load one of the NDWI input files to extract spatial reference
with rasterio.open(post_flood_files[0]) as src:
    transform = src.transform
    raster_crs = src.crs

# Define output shapefile path
output_shapefile = "/content/sentinel-2_flood_layer.shp"

# Save the flood layer
convert_flood_to_shapefile(flood_layer, transform, output_shapefile, raster_crs)
